# Tugas Praktikum 5: Sequential Pattern Analysis

Nama  : Levina Zahrathul Huda \
NRP   : 5054251033

In [97]:
import sys

!{sys.executable} -m pip install prefixspan

# 1. Unduh dan Import Dataset

In [127]:
import pandas as pd 
import numpy as np
import os
from prefixspan import PrefixSpan
from collections import Counter


In [99]:
df = pd.read_excel('Online Retail.xlsx')
print("Jumlah baris dan kolom:", df.shape)
df.info()
df.head()

Jumlah baris dan kolom: (541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# 2. Eksplorasi dan Pembersihan Data

In [100]:
print("Data sebelum cleaning")
print("Jumlah Baris:", len(df))
print("Jumlah Invoice:", df['InvoiceNo'].nunique())
print("Jumlah Pelanggan:", df['CustomerID'].nunique())

# Missing values
print("\nMissing values:")
print(df.isna().sum())

df[['CustomerID', 'InvoiceDate', 'InvoiceNo', 'Description', 'Quantity']].head()


Data sebelum cleaning
Jumlah Baris: 541909
Jumlah Invoice: 25900
Jumlah Pelanggan: 4372

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


,CustomerID,InvoiceDate,InvoiceNo,Description,Quantity
0,17850.0,2010-12-01 08:26:00,536365,WHITE HANGING HEART T-LIGHT HOLDER,6
1,17850.0,2010-12-01 08:26:00,536365,WHITE METAL LANTERN,6
2,17850.0,2010-12-01 08:26:00,536365,CREAM CUPID HEARTS COAT HANGER,8
3,17850.0,2010-12-01 08:26:00,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6
4,17850.0,2010-12-01 08:26:00,536365,RED WOOLLY HOTTIE WHITE HEART.,6


In [101]:
# Hapus baris dengan `Description` & `CustomerID` kosong
df_clean = df.dropna(subset=['Description', 'CustomerID'])

# Hapus `InvoiceNo` yang diawali huruf C
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Hapus baris dengan `Quantity` yang negatif atau 0
df_clean = df_clean[df_clean['Quantity'] > 0]

print("Data setelah cleaning")
print("Jumlah Baris:", len(df_clean))
print("Jumlah Invoice:", df_clean['InvoiceNo'].nunique())
print("Jumlah Pelanggan:", df_clean['CustomerID'].nunique())

Data setelah cleaning
Jumlah Baris: 397924
Jumlah Invoice: 18536
Jumlah Pelanggan: 4339


In [102]:
print("Perbandingan jumlah pelanggan sebelum dan sesudah cleaning")
print("Jumlah Pelanggan Sebelum Cleaning:", df['CustomerID'].nunique())
print("Jumlah Pelanggan Sesudah Cleaning:", df_clean['CustomerID'].nunique())

Perbandingan jumlah pelanggan sebelum dan sesudah cleaning
Jumlah Pelanggan Sebelum Cleaning: 4372
Jumlah Pelanggan Sesudah Cleaning: 4339


# 3. Persiapan Data

In [ ]:
# Mengurutkan data berdasarkan `CustomerID` & `InvoiceDate`
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean = df_clean.sort_values(['CustomerID', 'InvoiceDate', 'InvoiceNo'])

# Mengelompokkan item dalam satu invoice
invoice_data = df_clean.groupby(
    ['CustomerID', 'InvoiceNo']
).agg({
    'InvoiceDate': 'min',
    'Description': lambda x: sorted(set(x))
}).reset_index()

# Mengurutkan kembali data berdasarkan `CustomerId` & `InvoiceDate`
invoice_data = invoice_data.sort_values(['CustomerID', 'InvoiceDate', 'InvoiceNo'])

# Menyusun sequence setiap pelanggan
sequences = invoice_data.groupby('CustomerID')['Description'].apply(list).tolist() 

print("Jumlah sequence:", len(sequences))
for sequence in sequences[:5]:
    print(sequence)

Jumlah sequence: 4339
[['MEDIUM CERAMIC TOP STORAGE JAR']]
[['3D DOG PICTURE PLAYING CARDS', 'AIRLINE BAG VINTAGE JET SET BROWN', 'ALARM CLOCK BAKELIKE CHOCOLATE', 'ALARM CLOCK BAKELIKE GREEN', 'ALARM CLOCK BAKELIKE ORANGE', 'ALARM CLOCK BAKELIKE PINK', 'ALARM CLOCK BAKELIKE RED ', 'BATHROOM METAL SIGN ', 'BLACK CANDELABRA T-LIGHT HOLDER', 'BLACK EAR MUFF HEADPHONES', 'BLACK GRAND BAROQUE PHOTO FRAME', 'BLUE 3 PIECE POLKADOT CUTLERY SET', 'BLUE DRAWER KNOB ACRYLIC EDWARDIAN', 'BOOM BOX SPEAKER BOYS', 'BOX OF 6 ASSORTED COLOUR TEASPOONS', 'CAMOUFLAGE EAR MUFF HEADPHONES', 'CLEAR DRAWER KNOB ACRYLIC EDWARDIAN', 'COLOUR GLASS. STAR T-LIGHT HOLDER', 'EMERGENCY FIRST AID TIN ', 'FOUR HOOK  WHITE LOVEBIRDS', 'GREEN DRAWER KNOB ACRYLIC EDWARDIAN', 'LARGE HEART MEASURING SPOONS', 'MINI PAINT SET VINTAGE ', 'PINK 3 PIECE POLKADOT CUTLERY SET', 'PINK DRAWER KNOB ACRYLIC EDWARDIAN', 'PURPLE DRAWERKNOB ACRYLIC EDWARDIAN', 'RED 3 PIECE RETROSPOT CUTLERY SET', 'RED DRAWER KNOB ACRYLIC EDWARDIAN', 'R

# 4. Penerapan Algoritma

In [141]:
min_support = 0.02
min_support_count = int(np.ceil(min_support * len(sequences)))

# jumlah pelanggan yang membeli tiap produk
item_counts = Counter()

for sequence in sequences:
    for itemset in sequence:
        for item in itemset:
            item_counts[item] += 1

top_items = [
    item for item, count in item_counts.most_common(100)
    if count >= min_support_count
]

print("Jumlah produk unik:", len(item_counts))
print("Jumlah produk yang dianalisis:", len(top_items))
print("Minimum support:", min_support)
print("Minimum support count:", min_support_count)
print("\n10 produk teratas:")
print(item_counts.most_common(10))

Jumlah produk unik: 3877
Jumlah produk yang dianalisis: 100
Minimum support: 0.02
Minimum support count: 87

10 produk teratas:
[('WHITE HANGING HEART T-LIGHT HOLDER', 1971), ('REGENCY CAKESTAND 3 TIER', 1704), ('JUMBO BAG RED RETROSPOT', 1600), ('PARTY BUNTING', 1380), ('ASSORTED COLOUR BIRD ORNAMENT', 1375), ('LUNCH BAG RED RETROSPOT', 1288), ('SET OF 3 CAKE TINS PANTRY DESIGN ', 1146), ('POSTAGE', 1099), ('LUNCH BAG  BLACK SKULL.', 1052), ('PACK OF 72 RETROSPOT CAKE CASES', 1029)]


In [142]:
pattern_counts = Counter()

for sequence in sequences:
    previous_items = set()
    customer_patterns = set()

    for itemset in sequence:
        current_items = set(itemset) & set(top_items)

        for first_item in previous_items:
            for second_item in current_items:
                customer_patterns.add((first_item, second_item))

        previous_items.update(current_items)

    pattern_counts.update(customer_patterns)

patterns = [
    (count, pattern)
    for pattern, count in pattern_counts.items()
    if count >= min_support_count
]

print("Jumlah pola yang ditemukan:", len(patterns))
print("\nPola yang ditemukan:")
for support, pattern in patterns[:20]:
    print(pattern, "Support count:", support)

Jumlah pola yang ditemukan: 1123

Pola yang ditemukan:
('ALARM CLOCK BAKELIKE RED ', 'ALARM CLOCK BAKELIKE RED ') Support count: 188
('ALARM CLOCK BAKELIKE RED ', 'REGENCY CAKESTAND 3 TIER') Support count: 91
('REGENCY CAKESTAND 3 TIER', 'REGENCY CAKESTAND 3 TIER') Support count: 330
('WOODLAND CHARLOTTE BAG', 'WOODLAND CHARLOTTE BAG') Support count: 114
('ALARM CLOCK BAKELIKE GREEN', 'ALARM CLOCK BAKELIKE GREEN') Support count: 156
('ROSES REGENCY TEACUP AND SAUCER ', 'ROSES REGENCY TEACUP AND SAUCER ') Support count: 143
('REGENCY CAKESTAND 3 TIER', 'RABBIT NIGHT LIGHT') Support count: 131
('60 TEATIME FAIRY CAKE CASES', '60 TEATIME FAIRY CAKE CASES') Support count: 117
('REGENCY CAKESTAND 3 TIER', 'ROSES REGENCY TEACUP AND SAUCER ') Support count: 165
('REGENCY CAKESTAND 3 TIER', 'ALARM CLOCK BAKELIKE PINK') Support count: 99
('REGENCY CAKESTAND 3 TIER', 'ALARM CLOCK BAKELIKE RED ') Support count: 115
('ALARM CLOCK BAKELIKE PINK', 'ALARM CLOCK BAKELIKE GREEN') Support count: 132
('A

Pada percobaan pertama, digunakan minimum support sebesar 0,01 sesuai ketentuan. Hasil yang ditemukan yaitu sebesar 5.646 pola. Karena jumlah tersebut cukup banyak, minimum support disesuaikan dengan dinaikkan menjadi 0,02 agar pola yang dihasilkan lebih berfokus pada pola pembelian yang lebih sering muncul.

# 5. Evaluasi Pola

In [146]:
evaluasi = []

for support_count, pattern in patterns:
    first_item, second_item = pattern
    first_count = sum(
        1 for sequence in sequences
        if any(first_item in itemset for itemset in sequence)
    )
    support_value = support_count/len(sequences)
    confidence = support_count/first_count

    evaluasi.append({
        'Pola': f'{first_item} -> {second_item}',
        'Support count': support_count,
        'Support': support_value,
        'Confidence': confidence
    })

hasil_evaluasi = pd.DataFrame(evaluasi)
hasil_evaluasi = hasil_evaluasi.sort_values(
    by='Support',
    ascending=False
)

print(hasil_evaluasi.head(10))

                                                  Pola  Support count  \
346  WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HA...            405   
2    REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND ...            330   
235  JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETRO...            312   
344  ASSORTED COLOUR BIRD ORNAMENT -> ASSORTED COLO...            291   
485                     PARTY BUNTING -> PARTY BUNTING            280   
306  LUNCH BAG RED RETROSPOT -> LUNCH BAG RED RETRO...            276   
494  LUNCH BAG  BLACK SKULL. -> LUNCH BAG  BLACK SK...            229   
47   SET OF 3 CAKE TINS PANTRY DESIGN  -> SET OF 3 ...            225   
539  LUNCH BAG RED RETROSPOT -> LUNCH BAG  BLACK SK...            212   
260     LUNCH BAG CARS BLUE -> LUNCH BAG RED RETROSPOT            209   

      Support  Confidence  
346  0.093339    0.473131  
2    0.076054    0.374574  
235  0.071906    0.491339  
344  0.067066    0.429204  
485  0.064531    0.395480  
306  0.063609    0.518797  


In [147]:
top_3 = hasil_evaluasi.head(3)

print("Top 3 dengan support tertinggi:")
print(top_3.to_string(index=False))

Top 3 dengan support tertinggi:
                                                                    Pola  Support count  Support  Confidence
WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER            405 0.093339    0.473131
                    REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND 3 TIER            330 0.076054    0.374574
                      JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETROSPOT            312 0.071906    0.491339


Interpretasi dalam konteks bisnis ritel 
1. WHITE HANGING HEART T-LIGHT HOLDER -> WHITE HANGING HEART T-LIGHT HOLDER \
Pola ini ditemukan pada 405 pelanggan dengan support sebesar 9,33% dan confidence sebesar 47,31%. Dari pelanggan yang pernah membeli produk WHITE HANGING HEART T-LIGHT HOLDER, sekitar 47,31% nya memiliki pola pembelian produk yang sama pada transaksi berikutnya. Dalam bisnis ritel, hal ini dapat menunjukkan adanya kecenderungan pelanggan untuk membeli produk tersebut kembali.

2. REGENCY CAKESTAND 3 TIER -> REGENCY CAKESTAND 3 TIER \
Pola ini ditemukan pada sebanyak 330 pelanggan dengan support sebesar 7,61% dan confidence sebesar 37,46%. Sekitar 37,46% pelanggan yang pernah membeli REGENCY CAKESTAND 3 TIER juga menunjukkan pembelian produk yang sama pada transaksi berikutnya. Hal ini dapat menjadi indikasi adanya pembelian berulang, sehingga toko dapat mempertimbangkan produk ini sebagai salah satu produk yang perlu dipertahankan ketersediannya.

3. JUMBO BAG RED RETROSPOT -> JUMBO BAG RED RETROSPOT \
Pola ini ditemukan pada 312 pelanggan dengan support sebesar 7,19% dan confidence sebesar 49,31%. Hampir setengah dari pelanggan yang pernah membeli JUMBO BAG RED RETROSPOT memiliki pola pembelian produk yang sama pada transaksi berikutnya. Nilai confidence yang paling tinggi antara ketiga pola menunjukkan bahwa produk ini memiliki kecenderungan pembelian berulang yang relatif lebih kuat. Toko dapat mempertimbangkan untuk menjaga stok produknya dan dimanfaatkan dalam strategi pemasaran pelanggan.